# Compare CellViT Inference Runs (Tables + Disagreement Images)

This notebook:
1. Defines a `MODEL_RUNS` dict mapping model labels → run directories.
2. Runs inference for each run directory (optional / can skip if results already exist).
3. Loads each `inference_results.json` and builds:
   - Overall metrics table
   - Per-class PQ table
   - Per-class F1/Precision/Recall table
4. Computes "most different" images between two models (by `bPQ` or `Dice` deltas).

> Tip: run the **inference** step only when your training jobs are finished (or point to already-finished run dirs).


In [1]:
from __future__ import annotations

import json
import math
import os
import subprocess
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [2]:
# ----------------------------
# 1) Configure paths + runs
# ----------------------------
from datetime import datetime

# Set this to the AI-GUIDED-CLEAN root on SCC
ROOT_PATH = Path("/projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/AI-GUIDED-CLEAN")

# Output dir with timestamp so multiple runs don't overwrite
OUT_DIR = Path("./comparison_outputs") / datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
OUT_DIR.mkdir(parents=True, exist_ok=True)

INFER_SCRIPT = ROOT_PATH / "CellViT-plus-plus/cellvit/training/evaluate/inference_cellvit_experiment_pannuke.py"

# Edit this dict for your current experiments
MODEL_RUNS: Dict[str, Path] = {
    # "SAM-H Baseline (Prev)": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256_pannuke/logs_local/2025-12-05T174106_tcga_finetune_256",
    "SAM-H (New)": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local/2026-02-10T225437_tcga_finetune_256",
    #"FiLM Rosie Weights SamH Baseline": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256_pannuke/logs_local_berk/2026-02-11T233928_FilmRosieWeights-samhbaseline",
    #"FiLM": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256_pannuke/logs_local_berk/2026-02-10T225437_First_film try same with patch size 256",
    #"Virchow": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local_virchow/2026-02-10T225413_VirchowTrainTestValSplit",
    #"FiLM Rosie Weights": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local_berk/2026-02-10T231234_FilmRosieWeights",
    "FiLM Rosie Weights z1z4": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local_berk/2026-02-14T105042_FilmRosieWeights-z1z4",
    "FiLM Rosie Weights z3z4": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local_berk/2026-02-14T105042_FilmRosieWeights-z3z4",
    "FiLM Rosie Weights z4": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local_berk/2026-02-14T105042_FilmRosieWeights-z4",
    "FiLM Rosie Weights Baseline": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local_berk/2026-02-14T105042_FilmRosieWeights-samhbaseline",
    "FiLM IMAGENET Weights z1z4": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local_berk/2026-02-14T154156_FilmIMAGENETWeights-z1z4",
    "FiLM IMAGENET Weights z3z4": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local_berk/2026-02-14T154138_FilmIMAGENETWeights-z3z4",
    "FiLM IMAGENET Weights z4": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local_berk/2026-02-14T154204_FilmIMAGENETWeights-z4",
    "FiLM IMAGENET Weights Baseline": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local_berk/2026-02-14T154204_FilmIMAGENETWeights-samhbaseline",
    "Early Fusion map9c8 lr1e-4": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local_berk/2026-02-17T121656_EarlyFusion-map9c8-lr1e-4",
    "Early Fusion vec9 lr3e-05": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local_berk/2026-02-17T103313_EarlyFusion-vec9-lr3e-05",
    "Early Fusion vec9 lr1e-4": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local_berk/2026-02-17T103313_EarlyFusion-vec9-lr1e-4",
    "Early Fusion map9c8 lr3e-05": ROOT_PATH / "ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local_berk/2026-02-17T103313_EarlyFusion-map9c8-lr3e-05",
}

GPU_ID = 0

In [3]:
# ----------------------------
# 2) Helpers: run inference
# ----------------------------

def run_inference_for_run(run_dir: Path, gpu: int = 0, force: bool = False) -> Path:
    """Run inference script for a single run directory.
    Returns path to inference_results.json.
    """
    run_dir = Path(run_dir)
    out_json = run_dir / "inference_results.json"

    if out_json.exists() and not force:
        print(f"✅ exists, skipping: {out_json}")
        return out_json

    cmd = [
        "python",
        str(INFER_SCRIPT),
        "--run_dir", str(run_dir),
        "--gpu", str(gpu),
    ]

    print("▶️", " ".join(cmd))
    subprocess.run(cmd, check=True)
    if not out_json.exists():
        raise FileNotFoundError(f"Expected {out_json} but not found.")
    print(f"✅ wrote: {out_json}")
    return out_json


def run_all_inference(model_runs: Dict[str, Path], gpu: int = 0, force: bool = False) -> Dict[str, Path]:
    results = {}
    for name, rd in model_runs.items():
        print(f"\n=== {name} ===")
        results[name] = run_inference_for_run(rd, gpu=gpu, force=force)
    return results

In [4]:
# ----------------------------
# 3) Helpers: parse json → tables
# ----------------------------

def _safe_float(x):
    try:
        if x is None:
            return float("nan")
        if isinstance(x, str):
            # handle "nan"
            if x.lower() == "nan":
                return float("nan")
        return float(x)
    except Exception:
        return float("nan")


def load_inference_json(path: Path) -> dict:
    with open(path, "r") as f:
        return json.load(f)


def build_overall_table(results: Dict[str, dict]) -> pd.DataFrame:
    rows = []
    for model_name, d in results.items():
        ds = d.get("dataset", {})
        row = {"model": model_name}
        for k, v in ds.items():
            row[k] = _safe_float(v)
        rows.append(row)
    df = pd.DataFrame(rows).set_index("model")
    # nicer ordering (keep common metrics first if present)
    preferred = ["mPQ", "mDQ", "mSQ", "bPQ", "bDQ", "bSQ", "Binary-Cell-Dice-Mean", "Binary-Cell-Jacard-Mean",
                 "f1_detection", "precision_detection", "recall_detection", "Tissue-Multiclass-Accuracy"]
    cols = [c for c in preferred if c in df.columns] + [c for c in df.columns if c not in preferred]
    return df[cols].sort_index()


def build_perclass_pq_table(results: Dict[str, dict]) -> pd.DataFrame:
    rows = []
    for model_name, d in results.items():
        pq = d.get("nuclei_metrics_pq", {})
        row = {"model": model_name}
        for cls, val in pq.items():
            row[cls] = _safe_float(val)
        rows.append(row)
    return pd.DataFrame(rows).set_index("model").sort_index()


def build_perclass_detection_table(results: Dict[str, dict]) -> pd.DataFrame:
    # nuclei_metrics_d: {class: {f1_cell, prec_cell, rec_cell}}
    rows = []
    for model_name, d in results.items():
        det = d.get("nuclei_metrics_d", {})
        row = {"model": model_name}
        for cls, stats in det.items():
            if not isinstance(stats, dict):
                continue
            row[f"{cls}__f1"] = _safe_float(stats.get("f1_cell"))
            row[f"{cls}__prec"] = _safe_float(stats.get("prec_cell"))
            row[f"{cls}__rec"] = _safe_float(stats.get("rec_cell"))
        rows.append(row)
    df = pd.DataFrame(rows).set_index("model").sort_index()
    return df

In [5]:
# ----------------------------
# 4) Run inference for any model missing inference_results.json (subprocess — no model loaded in notebook)
# ----------------------------
# Each run is a separate subprocess: loads model, runs inference, writes JSON, exits. No models stay in notebook.
FORCE_RERUN = False
run_all_inference(MODEL_RUNS, gpu=GPU_ID, force=FORCE_RERUN)


=== SAM-H (New) ===
✅ exists, skipping: /projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/AI-GUIDED-CLEAN/ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local/2026-02-10T225437_tcga_finetune_256/inference_results.json

=== FiLM Rosie Weights z1z4 ===
✅ exists, skipping: /projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/AI-GUIDED-CLEAN/ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local_berk/2026-02-14T105042_FilmRosieWeights-z1z4/inference_results.json

=== FiLM Rosie Weights z3z4 ===
✅ exists, skipping: /projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/AI-GUIDED-CLEAN/ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local_berk/2026-02-14T105042_FilmRosieWeights-z3z4/inference_results.json

=== FiLM Rosie Weights z4 ===
✅ exists, skipping: /projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-sli

{'SAM-H (New)': PosixPath('/projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/AI-GUIDED-CLEAN/ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local/2026-02-10T225437_tcga_finetune_256/inference_results.json'),
 'FiLM Rosie Weights z1z4': PosixPath('/projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/AI-GUIDED-CLEAN/ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local_berk/2026-02-14T105042_FilmRosieWeights-z1z4/inference_results.json'),
 'FiLM Rosie Weights z3z4': PosixPath('/projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/AI-GUIDED-CLEAN/ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local_berk/2026-02-14T105042_FilmRosieWeights-z3z4/inference_results.json'),
 'FiLM Rosie Weights z4': PosixPath('/projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/AI-GUIDED-CLEAN/ProcessedDa

In [6]:
# ----------------------------
# 5) Load jsons + build tables
# ----------------------------
def load_all_results(model_runs: Dict[str, Path]) -> Dict[str, dict]:
    out = {}
    for name, rd in model_runs.items():
        p = rd / "inference_results.json"
        if not p.exists():
            raise FileNotFoundError(f"Missing {p}. Run inference first.")
        out[name] = load_inference_json(p)
    return out

results = load_all_results(MODEL_RUNS)

overall_df = build_overall_table(results)
perclass_pq_df = build_perclass_pq_table(results)
perclass_det_df = build_perclass_detection_table(results)

display(overall_df)

,mPQ,mDQ,mSQ,bPQ,bDQ,bSQ,Binary-Cell-Dice-Mean,Binary-Cell-Jacard-Mean,f1_detection,precision_detection,recall_detection,Tissue-Multiclass-Accuracy
model,,,,,,,,,,,,
Early Fusion map9c8 lr1e-4,0.544318,0.663721,0.676933,0.636933,0.776417,0.788540,0.760572,0.652438,0.835994,0.835509,0.836480,1.0
Early Fusion map9c8 lr3e-05,0.548298,0.661349,0.731394,0.657557,0.794933,0.810895,0.781893,0.672649,0.838627,0.819341,0.858844,1.0
Early Fusion vec9 lr1e-4,0.557498,0.679708,0.705498,0.659197,0.805633,0.815476,0.788394,0.675539,0.834496,0.818080,0.851583,1.0
Early Fusion vec9 lr3e-05,0.575347,0.691942,0.720563,0.660660,0.793055,0.815735,0.780017,0.670805,0.827779,0.840671,0.815277,1.0
FiLM IMAGENET Weights Baseline,0.579894,0.701391,0.730331,0.665227,0.803351,0.819077,0.797771,0.688747,0.829926,0.828605,0.831252,1.0
FiLM IMAGENET Weights z1z4,0.558080,0.675919,0.702241,0.656999,0.794834,0.801984,0.786809,0.677493,0.831060,0.830577,0.831542,1.0
FiLM IMAGENET Weights z3z4,0.552187,0.665171,0.688436,0.663134,0.800662,0.818129,0.789556,0.683418,0.837002,0.829247,0.844903,1.0
FiLM IMAGENET Weights z4,0.569389,0.686630,0.716776,0.655707,0.791932,0.808777,0.782634,0.673163,0.841297,0.838868,0.843741,1.0
FiLM Rosie Weights Baseline,0.586468,0.710377,0.729213,0.667485,0.804730,0.819459,0.787854,0.678323,0.828625,0.818414,0.839094,1.0


## 5b) Styled tables (highlight winners)

These displays highlight:
- **Max value per column** (best metric / best class)
- **Best model row** by a chosen primary metric (e.g., `bPQ` or `mPQ`)


In [7]:
import numpy as np

# Choose a primary metric to define the "best overall" model row highlight.
# Common choices: "bPQ" (binary PQ) or "mPQ" (multi-class PQ)
PRIMARY_METRIC = "bPQ"


def style_highlight_max_per_column(df: pd.DataFrame, primary_metric: str | None = None):
    """Return a pandas Styler that highlights/bolds max per column and (optionally) best row."""
    # base formatting + column-wise max highlight
    sty = (
        df.style
        .format(precision=4)
        .highlight_max(axis=0)
    )

    # bold the max entries per column
    def bold_max(s):
        vals = pd.to_numeric(s, errors="coerce")
        m = np.nanmax(vals.values)
        return ["font-weight: bold;" if (pd.notna(v) and float(v) == m) else "" for v in vals]

    sty = sty.apply(bold_max, axis=0)

    # highlight best row by primary metric
    if primary_metric is not None and primary_metric in df.columns:
        vals = pd.to_numeric(df[primary_metric], errors="coerce")
        if vals.notna().any():
            best_model = vals.idxmax()

            def highlight_best_row(row):
                return ["background-color: rgba(0, 200, 0, 0.12);" if row.name == best_model else "" for _ in row]

            sty = sty.apply(highlight_best_row, axis=1)

    return sty


# Styled overall metrics
display(style_highlight_max_per_column(overall_df, primary_metric=PRIMARY_METRIC))


,mPQ,mDQ,mSQ,bPQ,bDQ,bSQ,Binary-Cell-Dice-Mean,Binary-Cell-Jacard-Mean,f1_detection,precision_detection,recall_detection,Tissue-Multiclass-Accuracy
model,,,,,,,,,,,,
Early Fusion map9c8 lr1e-4,0.5443,0.6637,0.6769,0.6369,0.7764,0.7885,0.7606,0.6524,0.8360,0.8355,0.8365,1.0000
Early Fusion map9c8 lr3e-05,0.5483,0.6613,0.7314,0.6576,0.7949,0.8109,0.7819,0.6726,0.8386,0.8193,0.8588,1.0000
Early Fusion vec9 lr1e-4,0.5575,0.6797,0.7055,0.6592,0.8056,0.8155,0.7884,0.6755,0.8345,0.8181,0.8516,1.0000
Early Fusion vec9 lr3e-05,0.5753,0.6919,0.7206,0.6607,0.7931,0.8157,0.7800,0.6708,0.8278,0.8407,0.8153,1.0000
FiLM IMAGENET Weights Baseline,0.5799,0.7014,0.7303,0.6652,0.8034,0.8191,0.7978,0.6887,0.8299,0.8286,0.8313,1.0000
FiLM IMAGENET Weights z1z4,0.5581,0.6759,0.7022,0.6570,0.7948,0.8020,0.7868,0.6775,0.8311,0.8306,0.8315,1.0000
FiLM IMAGENET Weights z3z4,0.5522,0.6652,0.6884,0.6631,0.8007,0.8181,0.7896,0.6834,0.8370,0.8292,0.8449,1.0000
FiLM IMAGENET Weights z4,0.5694,0.6866,0.7168,0.6557,0.7919,0.8088,0.7826,0.6732,0.8413,0.8389,0.8437,1.0000
FiLM Rosie Weights Baseline,0.5865,0.7104,0.7292,0.6675,0.8047,0.8195,0.7879,0.6783,0.8286,0.8184,0.8391,1.0000


In [8]:
# Styled per-class PQ
display(
    perclass_pq_df.style
    .format(precision=4)
    .highlight_max(axis=0)
    .apply(lambda s: ["font-weight: bold;" if pd.notna(v) and float(v) == float(pd.to_numeric(s, errors='coerce').max()) else "" for v in s], axis=0)
)

# Styled per-class detection metrics (F1 / Precision / Recall)
display(
    perclass_det_df.style
    .format(precision=4)
    .highlight_max(axis=0)
    .apply(lambda s: ["font-weight: bold;" if pd.notna(v) and float(v) == float(pd.to_numeric(s, errors='coerce').max()) else "" for v in s], axis=0)
)


,epithelial,lymphocyte,macrophage,neutrophil,other
model,,,,,
Early Fusion map9c8 lr1e-4,0.6104,0.4650,0.1940,0.2128,nan
Early Fusion map9c8 lr3e-05,0.5809,0.5297,0.2121,0.3404,nan
Early Fusion vec9 lr1e-4,0.6026,0.4623,0.2808,0.2722,nan
Early Fusion vec9 lr3e-05,0.6150,0.4841,0.2963,0.2474,nan
FiLM IMAGENET Weights Baseline,0.6225,0.5342,0.2812,0.3423,nan
FiLM IMAGENET Weights z1z4,0.6208,0.4795,0.2183,0.4116,nan
FiLM IMAGENET Weights z3z4,0.6240,0.4507,0.2095,0.3210,nan
FiLM IMAGENET Weights z4,0.6210,0.4795,0.2713,0.3108,nan
FiLM Rosie Weights Baseline,0.6247,0.4719,0.3357,0.4138,nan


,epithelial__f1,epithelial__prec,epithelial__rec,lymphocyte__f1,lymphocyte__prec,lymphocyte__rec,macrophage__f1,macrophage__prec,macrophage__rec,neutrophil__f1,neutrophil__prec,neutrophil__rec,other__f1,other__prec,other__rec
model,,,,,,,,,,,,,,,
Early Fusion map9c8 lr1e-4,0.8159,0.8305,0.8017,0.8107,0.7720,0.8536,0.4302,0.6981,0.3109,0.2812,0.5000,0.1957,nan,nan,nan
Early Fusion map9c8 lr3e-05,0.7456,0.8342,0.6740,0.7478,0.6416,0.8961,0.3778,0.7391,0.2537,0.3208,0.2500,0.4474,nan,nan,nan
Early Fusion vec9 lr1e-4,0.8106,0.8193,0.8022,0.8064,0.7597,0.8591,0.4976,0.5714,0.4407,0.5000,0.5143,0.4865,nan,nan,nan
Early Fusion vec9 lr3e-05,0.8008,0.8272,0.7760,0.8152,0.7963,0.8350,0.5414,0.7424,0.4261,0.5517,0.7619,0.4324,nan,nan,nan
FiLM IMAGENET Weights Baseline,0.8031,0.8198,0.7871,0.8226,0.8015,0.8448,0.4167,0.5000,0.3571,0.3800,0.2754,0.6129,nan,nan,nan
FiLM IMAGENET Weights z1z4,0.8162,0.8265,0.8062,0.8198,0.7979,0.8430,0.3388,0.6200,0.2331,0.3636,0.2632,0.5882,nan,nan,nan
FiLM IMAGENET Weights z3z4,0.8100,0.8259,0.7947,0.8179,0.7760,0.8646,0.3560,0.6415,0.2464,0.3717,0.2625,0.6364,nan,nan,nan
FiLM IMAGENET Weights z4,0.8187,0.8255,0.8121,0.8324,0.8050,0.8617,0.4889,0.7097,0.3729,0.5667,0.6296,0.5152,nan,nan,nan
FiLM Rosie Weights Baseline,0.8090,0.8143,0.8037,0.8146,0.7772,0.8557,0.5579,0.7361,0.4492,0.5970,0.5882,0.6061,nan,nan,nan


In [9]:
# Optional: export styled overall table to HTML (keeps highlights/bold)
html_path = OUT_DIR / "overall_table_styled.html"
(style_highlight_max_per_column(overall_df, primary_metric=PRIMARY_METRIC)
 .to_html(html_path))

print("Wrote:", html_path)


Wrote: comparison_outputs/2026-02-17_20-40-39/overall_table_styled.html


In [10]:
# Per-class PQ
display(perclass_pq_df)

# Per-class detection metrics (F1/Prec/Rec)
display(perclass_det_df)

,epithelial,lymphocyte,macrophage,neutrophil,other
model,,,,,
Early Fusion map9c8 lr1e-4,0.610428,0.465023,0.193996,0.212823,NaN
Early Fusion map9c8 lr3e-05,0.580878,0.529697,0.212100,0.340411,NaN
Early Fusion vec9 lr1e-4,0.602601,0.462326,0.280788,0.272240,NaN
Early Fusion vec9 lr3e-05,0.614983,0.484095,0.296344,0.247433,NaN
FiLM IMAGENET Weights Baseline,0.622498,0.534163,0.281182,0.342306,NaN
FiLM IMAGENET Weights z1z4,0.620765,0.479500,0.218324,0.411647,NaN
FiLM IMAGENET Weights z3z4,0.624047,0.450717,0.209477,0.321023,NaN
FiLM IMAGENET Weights z4,0.620986,0.479543,0.271349,0.310813,NaN
FiLM Rosie Weights Baseline,0.624667,0.471933,0.335719,0.413833,NaN


,epithelial__f1,epithelial__prec,epithelial__rec,lymphocyte__f1,lymphocyte__prec,lymphocyte__rec,macrophage__f1,macrophage__prec,macrophage__rec,neutrophil__f1,neutrophil__prec,neutrophil__rec,other__f1,other__prec,other__rec
model,,,,,,,,,,,,,,,
Early Fusion map9c8 lr1e-4,0.815877,0.830527,0.801733,0.810742,0.771961,0.853626,0.430233,0.698113,0.310924,0.281250,0.500000,0.195652,NaN,NaN,NaN
Early Fusion map9c8 lr3e-05,0.745575,0.834158,0.674000,0.747763,0.641555,0.896113,0.377778,0.739130,0.253731,0.320755,0.250000,0.447368,NaN,NaN,NaN
Early Fusion vec9 lr1e-4,0.810649,0.819330,0.802151,0.806361,0.759694,0.859136,0.497608,0.571429,0.440678,0.500000,0.514286,0.486486,NaN,NaN,NaN
Early Fusion vec9 lr3e-05,0.800783,0.827168,0.776030,0.815200,0.796320,0.834997,0.541436,0.742424,0.426087,0.551724,0.761905,0.432432,NaN,NaN,NaN
FiLM IMAGENET Weights Baseline,0.803080,0.819764,0.787062,0.822575,0.801517,0.844770,0.416667,0.500000,0.357143,0.380000,0.275362,0.612903,NaN,NaN,NaN
FiLM IMAGENET Weights z1z4,0.816203,0.826451,0.806206,0.819799,0.797859,0.842981,0.338798,0.620000,0.233083,0.363636,0.263158,0.588235,NaN,NaN,NaN
FiLM IMAGENET Weights z3z4,0.809995,0.825868,0.794720,0.817924,0.776048,0.864576,0.356021,0.641509,0.246377,0.371681,0.262500,0.636364,NaN,NaN,NaN
FiLM IMAGENET Weights z4,0.818729,0.825511,0.812059,0.832370,0.804969,0.861702,0.488889,0.709677,0.372881,0.566667,0.629630,0.515152,NaN,NaN,NaN
FiLM Rosie Weights Baseline,0.808976,0.814325,0.803698,0.814557,0.777174,0.855718,0.557895,0.736111,0.449153,0.597015,0.588235,0.606061,NaN,NaN,NaN


In [11]:
# Save tables to disk (optional)
overall_df.to_csv(OUT_DIR / "model_overall_metrics.csv")
perclass_pq_df.to_csv(OUT_DIR / "perclass_pq.csv")
perclass_det_df.to_csv(OUT_DIR / "perclass_detection_metrics.csv")

print("Wrote:", OUT_DIR)

Wrote: comparison_outputs/2026-02-17_20-40-39


In [12]:
# ----------------------------
# 6) Image-level disagreement
# ----------------------------
# Each inference json has image_metrics: {image_id: {Dice, Jaccard, bPQ, ...}}
# We'll compute top-K images where modelA and modelB differ most by a chosen metric.

def image_metric_df(d: dict, metric: str = "bPQ") -> pd.DataFrame:
    im = d.get("image_metrics", {})
    rows = []
    for image_id, m in im.items():
        if isinstance(m, dict) and metric in m:
            rows.append({"image_id": image_id, metric: _safe_float(m[metric])})
    return pd.DataFrame(rows).set_index("image_id")

def top_disagreement(modelA: str, modelB: str, metric: str = "bPQ", k: int = 20) -> pd.DataFrame:
    a = image_metric_df(results[modelA], metric=metric)
    b = image_metric_df(results[modelB], metric=metric)
    df = a.join(b, lsuffix=f"__{modelA}", rsuffix=f"__{modelB}", how="inner")
    df["delta"] = (df[f"{metric}__{modelB}"] - df[f"{metric}__{modelA}"]).abs()
    df["signed_delta"] = (df[f"{metric}__{modelB}"] - df[f"{metric}__{modelA}"])
    df = df.sort_values("delta", ascending=False).head(k)
    return df

BASELINE_NAME = "SAM-H (New)"
COMPARE_NAME = "FiLM Rosie Weights"  # change to any model label
METRIC = "bPQ"

disagree_df = top_disagreement(BASELINE_NAME, COMPARE_NAME, metric=METRIC, k=30)
display(disagree_df)

KeyError: 'FiLM Rosie Weights'

## Patch visualization (GT vs models)

**Step 1 (below):** Save all disagreement plots to disk (avoids memory issues).
**Step 2 (dropdown cell):** Browse saved plots with a lightweight dropdown (no model loading).

Folder structure: `comparison_outputs/disagreement_plots/{baseline}_vs_{compare}_{metric}/`
- `rank001_{image_id}.png` — main comparison figure
- `rank001_{image_id}_legend.png` — cell type legend
- `manifest.json` — rank, image_id, delta for dropdown labels


In [ ]:
# ----------------------------
# Setup for patch visualization (uses inference_utils)
# ----------------------------
import re
import sys
sys.path.insert(0, str(ROOT_PATH / "CellViT-plus-plus"))

import numpy as np
from PIL import Image

from inference_utils import (
    load_cellvit_inference_single_patch,
    build_postprocess_trainer,
    infer_single,
    build_lut_from_dataset_config,
    visualize_type_overlay_error_matrix,
)

# Get dataset path from first model run's config
import yaml
_first_run = list(MODEL_RUNS.values())[0]
with open(_first_run / "config.yaml") as f:
    _conf = yaml.safe_load(f)
DATASET_PATH = Path(_conf["data"]["dataset_path"])
IMAGES_DIR = DATASET_PATH / "images"
PATCH_LABEL_MAPS_DIR = DATASET_PATH / "patch_label_maps"


def _sanitize_for_path(name: str) -> str:
    """Convert model name to safe folder/filename segment."""
    return re.sub(r"[^\w\-]", "_", str(name).strip()).strip("_") or "model"


def find_image_path(image_id: str) -> Path:
    stem = str(image_id).replace(".png", "")
    for base in [IMAGES_DIR, *[DATASET_PATH / f"fold{i}" / "images" for i in range(3)]]:
        for suffix in [".png", ""]:
            p = base / f"{stem}{suffix}"
            if p.exists():
                return p
    return IMAGES_DIR / f"{stem}.png"  # fallback


def find_gt_map_path(image_id: str) -> Path:
    stem = str(image_id).replace(".png", "")
    return PATCH_LABEL_MAPS_DIR / f"{stem}.npy"


# Plots output folder: OUT_DIR/disagreement_plots/{baseline}_vs_{compare}_{metric}/
PLOTS_BASE = OUT_DIR / "disagreement_plots"
PLOTS_SUBDIR = f"{_sanitize_for_path(BASELINE_NAME)}_vs_{_sanitize_for_path(COMPARE_NAME)}_{METRIC}"
PLOTS_DIR = PLOTS_BASE / PLOTS_SUBDIR

# Image list from disagree_df (ordered by delta descending)
IMAGE_OPTIONS = [str(x) for x in disagree_df.index.tolist()]

NameError: name 'ROOT_PATH' is not defined

In [ ]:
# ----------------------------
# Save disagreement plots one image at a time (no model caching to avoid OOM)
# Load one model → infer on image → release → next model. Same pattern as inference_notebook.
# ----------------------------
import gc

SKIP_EXISTING = True  # Set False to overwrite all

PLOTS_DIR.mkdir(parents=True, exist_ok=True)

_device = "cuda" if __import__("torch").cuda.is_available() else "cpu"
manifest = []

for rank, (image_id, row) in enumerate(disagree_df.iterrows(), start=1):
    image_id = str(image_id)
    stem = image_id.replace(".png", "")
    img_path = find_image_path(image_id)
    gt_path = find_gt_map_path(image_id)

    if not img_path.exists():
        print(f"⚠️ Skipping (not found): {image_id}")
        continue
    if not gt_path.exists():
        print(f"⚠️ Skipping (no GT): {image_id}")
        continue

    img = Image.open(img_path).convert("RGB")
    img_np = np.array(img)
    maps = np.load(gt_path, allow_pickle=True).item()
    gt_type_map = maps["type_map"].astype(np.int32)

    # Load ONE model at a time, infer, release — avoids holding multiple huge models
    pred_maps = {}
    dataset_config = None
    for name, run_dir in MODEL_RUNS.items():
        model, transforms, dataset_config, inference = load_cellvit_inference_single_patch(
            run_dir, magnification=40, gpu=GPU_ID, device=_device
        )
        trainer = build_postprocess_trainer(
            model, inference, dataset_config, run_dir,
            device=_device, magnification=40
        )
        out = infer_single(img_np, model, transforms, trainer, device=_device)
        pred_maps[name] = out["type_map"]
        del model, trainer, inference, transforms
        gc.collect()
        if _device == "cuda":
            import torch
            torch.cuda.empty_cache()
    lut, id_to_name, max_cls_id = build_lut_from_dataset_config(dataset_config)

    delta = float(row.get("delta", 0))
    fname = f"rank{rank:03d}_{stem}.png"
    save_path = PLOTS_DIR / fname
    if SKIP_EXISTING and save_path.exists():
        manifest.append({"rank": rank, "image_id": image_id, "delta": delta, "file": fname})
        print(f"⏭️ Skipped (exists): {fname}")
        continue
    visualize_type_overlay_error_matrix(
        gt_type_map, pred_maps, img_np, lut, id_to_name, max_cls_id,
        title=f"GT vs Models — {image_id} (Δ={delta:.3f})",
        save_path=save_path,
    )
    manifest.append({"rank": rank, "image_id": image_id, "delta": delta, "file": fname})
    print(f"✅ Saved {fname}")

    del img_np, gt_type_map, pred_maps
    gc.collect()

# Save manifest for dropdown
with open(PLOTS_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print(f"\n📁 Plots saved to: {PLOTS_DIR}")

Loaded run: /projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/AI-GUIDED-CLEAN/ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local/2026-02-10T225437_tcga_finetune_256
Loading best model from /projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/AI-GUIDED-CLEAN/ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local/2026-02-10T225437_tcga_finetune_256/checkpoints/model_best.pth
<All keys matched successfully>
Loaded run: /projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/AI-GUIDED-CLEAN/ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local_virchow/2026-02-10T225413_VirchowTrainTestValSplit
No checkpoint provided!
Loading best model from /projectnb/ec500kb/projects/Fall_2025_Projects/Project_2/AI-guided-whole-slide-imaging-analysis/AI-GUIDED-CLEAN/ProcessedDataset/v1_40x_area20_2/patches_cellvit_p256/logs_local_virchow/2026-0

KeyboardInterrupt: 

In [ ]:
# ----------------------------
# Browse saved plots with dropdown (lightweight, no model loading)
# ----------------------------
import ipywidgets as widgets
from IPython.display import display, clear_output, Image as IPyImage

# Uses PLOTS_DIR from setup cell above; or set explicitly if running this cell alone:
# PLOTS_BASE = OUT_DIR / "disagreement_plots"
# PLOTS_DIR = PLOTS_BASE / f"{_sanitize_for_path(BASELINE_NAME)}_vs_{_sanitize_for_path(COMPARE_NAME)}_{METRIC}"

manifest_path = PLOTS_DIR / "manifest.json"
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    options = [(f"#{m['rank']:03d} {m['image_id']} (Δ={m['delta']:.3f})", str(PLOTS_DIR / m["file"])) for m in manifest]
else:
    # Fallback: list PNGs (exclude _legend.png)
    pngs = sorted(p for p in PLOTS_DIR.glob("rank*.png") if "_legend" not in p.name)
    options = [(p.name, str(p)) for p in pngs]

dropdown = widgets.Dropdown(
    options=options,
    value=options[0][1] if options else None,
    description="Image:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="600px"),
)
output = widgets.Output()

def on_select(change):
    with output:
        clear_output(wait=True)
        if change["new"]:
            display(IPyImage(filename=change["new"], width=800))

# Unobserve before observe to avoid duplicate callbacks when cell is re-run
try:
    dropdown.unobserve(on_select, names="value")
except Exception:
    pass
dropdown.observe(on_select, names="value")
display(widgets.VBox([dropdown, output]))
if options:
    on_select({"new": options[0][1]})

NameError: name '_sanitize_for_path' is not defined